# CIFAR-10: Object Recognition
___


## Dependencies

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.linalg import vector_norm

import torch
from torch import nn

from einops import rearrange, repeat
from einops.layers.torch import Rearrange

In [4]:
def vectorize(x, multichannel=False):
    """Vectorize data in any shape.

    Args:
        x (torch.Tensor): input data
        multichannel (bool, optional): whether to keep the multiple channels (in the second dimension). Defaults to False.

    Returns:
        torch.Tensor: data of shape (sample_size, dimension) or (sample_size, num_channel, dimension) if multichannel is True.
    """
    if len(x.shape) == 1:
        return x.unsqueeze(1)
    if len(x.shape) == 2:
        return x
    else:
        if not multichannel: # one channel
            return x.reshape(x.shape[0], -1)
        else: # multi-channel
            return x.reshape(x.shape[0], x.shape[1], -1)

In [16]:
def energy_loss(x_true, x_est, beta=1, verbose=False):
    """Loss function based on the energy score.

    Args:
        x_true (torch.Tensor): iid samples from the true distribution of shape (data_size, data_dim)
        x_est (list of torch.Tensor): 
            - a list of length sample_size, where each element is a tensor of shape (data_size, data_dim) that contains one sample for each data point from the estimated distribution, or 
            - a tensor of shape (data_size*sample_size, response_dim) such that x_est[data_size*(i-1):data_size*i,:] contains one sample for each data point, for i = 1, ..., sample_size.
        beta (float): power parameter in the energy score.
        verbose (bool): whether to return two terms of the loss.

    Returns:
        loss (torch.Tensor): energy loss.
    """
    EPS = 0 if float(beta).is_integer() else 1e-5
    x_true = vectorize(x_true).unsqueeze(1)
    if not isinstance(x_est, list):
        x_est = list(torch.split(x_est, x_true.shape[0], dim=0))
    m = len(x_est)
    x_est = [vectorize(x_est[i]).unsqueeze(1) for i in range(m)]
    x_est = torch.cat(x_est, dim=1)
        
    s1 = (vector_norm(x_est - x_true, 2, dim=2) + EPS).pow(beta).mean()
    s2 = (torch.cdist(x_est, x_est, 2) + EPS).pow(beta).mean() * m / (m - 1)
    if verbose:
        return torch.cat([(s1 - s2 / 2).reshape(1), s1.reshape(1), s2.reshape(1)], dim=0)
    else:
        return (s1 - s2 / 2)

## Loading the Data

In [2]:
train_dataset = datasets.CIFAR10('./data', train=True, download=True,  # Downloads into a directory ../data
                               transform=transforms.ToTensor())
train_dataset, valid_dataset = torch.utils.data.random_split(train_dataset, 
                                                             [int(len(train_dataset)*0.8), int(len(train_dataset)*0.2)], 
                                                             generator=torch.Generator().manual_seed(42))
test_dataset = datasets.CIFAR10('./data', train=False, download=False,  # No need to download again
                              transform=transforms.ToTensor())

100%|████████████████████████████████████████████████████████████████████████████████████████| 170498071/170498071 [00:04<00:00, 34949382.59it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data


In [5]:
def run_training_loop(model, batch_size=32, n_epochs=10, lr=1e-3):
    """
    Run a training loop based on the input model and associated parameters
    
    Parameters:
        model: The input model to be trained
        batch_size: Number of training points to include in batch
        n_epochs: Number of epochs to train the model for
        lr: Learning rate used in Adam optimizer
        
    """
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=True)
    
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # Choose Adam as the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Use the cross entropy loss function
    loss_fn = nn.CrossEntropyLoss()

    # store metrics
    train_loss_history = np.zeros([n_epochs, 1])
    valid_accuracy_history = np.zeros([n_epochs, 1])
    valid_loss_history = np.zeros([n_epochs, 1])

    for epoch in range(n_epochs):

        # Some layers, such as Dropout, behave differently during training
        model.train()

        train_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            data, target = data.to(device), target.to(device)

            # Erase accumulated gradients
            optimizer.zero_grad()

            # Forward pass
            output = model(data)

            # Calculate loss
            loss = loss_fn(output, target)
            train_loss += loss.item()

            # Backward pass
            loss.backward()
            
            # Weight update
            optimizer.step()

        train_loss_history[epoch] = train_loss / len(train_loader.dataset)

        # Track loss each epoch
        print('Train Epoch: %d  Average loss: %.4f' %
              (epoch + 1,  train_loss_history[epoch]))

        # Putting layers like Dropout into evaluation mode
        model.eval()

        valid_loss = 0
        correct = 0

        # Turning off automatic differentiation
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                valid_loss += loss_fn(output, target).item()  # Sum up batch loss
                pred = output.argmax(dim=1, keepdim=True)  # Get the index of the max class score
                correct += pred.eq(target.view_as(pred)).sum().item()

        valid_loss_history[epoch] = valid_loss / len(valid_loader.dataset)
        valid_accuracy_history[epoch] = correct / len(valid_loader.dataset)

        print('Valid set: Average loss: %.4f, Accuracy: %d/%d (%.4f)\n' %
              (valid_loss_history[epoch], correct, len(valid_loader.dataset),
              100. * valid_accuracy_history[epoch]))
    
    return model, train_loss_history, valid_loss_history, valid_accuracy_history

In [32]:
def run_training_loop_energy(model, batch_size=32, n_epochs=10, lr=1e-3):
    """
    Run a training loop based on the input model and associated parameters
    
    Parameters:
        model: The input model to be trained
        batch_size: Number of training points to include in batch
        n_epochs: Number of epochs to train the model for
        lr: Learning rate used in Adam optimizer
        
    """
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=True)
    
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model.to(device)

    # Choose Adam as the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Use the cross entropy loss function
    loss_fn = nn.CrossEntropyLoss()

    # store metrics
    train_loss_history = np.zeros([n_epochs, 1])
    valid_accuracy_history = np.zeros([n_epochs, 1])
    valid_loss_history = np.zeros([n_epochs, 1])

    for epoch in range(n_epochs):

        # Some layers, such as Dropout, behave differently during training
        model.train()

        train_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            
            data, target = data.to(device), target.to(device)
            
            noise = torch.randn(
                        data.size(0),       # B
                        2,                  # 1 new channel
                        data.size(2),       # H
                        data.size(3),       # W
                        device=device
                    )
                    # now data is (B, C+1, H, W)
            data1 = torch.cat([data, noise[:, 0:1, :, :]], dim=1)
            data2 = torch.cat([data, noise[:, 1:2, :, :]], dim=1)

            # Erase accumulated gradients
            optimizer.zero_grad()

            # Forward pass
            output1 = model(data1)
            output2 = model(data2)
            
            loss = 0

            # Calculate loss
            loss1 = loss_fn(output1, target)
            loss2 = loss_fn(output2, target)
            
            loss += loss1
            loss += loss2
            
            train_loss += loss1.item()/2
            train_loss += loss2.item()/2
            
            # train_loss += energy_loss(output1, output2)
            # train_loss += energy_loss(output2, output1)
            
            loss += 0.01*energy_loss(output1, output2)
            loss += 0.01*energy_loss(output2, output1)
            
            

            # Backward pass
            loss.backward()
            
            # Weight update
            optimizer.step()

        train_loss_history[epoch] = train_loss / len(train_loader.dataset)

        # Track loss each epoch
        print('Train Epoch: %d  Average loss: %.4f' %
              (epoch + 1,  train_loss_history[epoch]))

        # Putting layers like Dropout into evaluation mode
        model.eval()

        valid_loss = 0
        correct = 0

        # Turning off automatic differentiation
        with torch.no_grad():
            for data, target in valid_loader:
                data, target = data.to(device), target.to(device)
                
                noise = torch.randn(
                        data.size(0),       # B
                        1,                  # 1 new channel
                        data.size(2),       # H
                        data.size(3),       # W
                        device=device
                    )
                    # now data is (B, C+1, H, W)
                data = torch.cat([data, noise[:, 0:1, :, :]], dim=1)
            
            
                output = model(data)
                valid_loss += loss_fn(output, target).item()  # Sum up batch loss
                pred = output.argmax(dim=1, keepdim=True)  # Get the index of the max class score
                correct += pred.eq(target.view_as(pred)).sum().item()

        valid_loss_history[epoch] = valid_loss / len(valid_loader.dataset)
        valid_accuracy_history[epoch] = correct / len(valid_loader.dataset)

        print('Valid set: Average loss: %.4f, Accuracy: %d/%d (%.4f)\n' %
              (valid_loss_history[epoch], correct, len(valid_loader.dataset),
              100. * valid_accuracy_history[epoch]))
    
    return model, train_loss_history, valid_loss_history, valid_accuracy_history

In [8]:
def test_performance(model, batch_size=32):
    """
    Test model performance on test dataset
    
    Parameters:
        model: The model to be tested
        batch_size: Number of training points to include in batch
    """
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True) 

    # Putting layers like Dropout into evaluation mode
    model.eval()
    # Use the cross entropy loss function
    loss_fn = nn.CrossEntropyLoss()
    
    # Send model to appropriate device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model.to(device)

    test_loss = 0
    correct = 0

    # Turning off automatic differentiation
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_fn(output, target).item()  # Sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # Get the index of the max class score
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = correct / len(test_loader.dataset)

    print('Test set: Average loss: %.4f, Accuracy: %d/%d (%.4f)' %
          (test_loss, correct, len(test_loader.dataset),
          100. * test_accuracy))
    return test_loss, test_accuracy

In [30]:
def test_performance_energy(model, batch_size=32):
    """
    Test model performance on test dataset
    
    Parameters:
        model: The model to be tested
        batch_size: Number of training points to include in batch
    """
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True) 

    # Putting layers like Dropout into evaluation mode
    model.eval()
    # Use the cross entropy loss function
    loss_fn = nn.CrossEntropyLoss()
    
    # Send model to appropriate device
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model.to(device)

    test_loss = 0
    correct = 0

    # Turning off automatic differentiation
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            noise = torch.randn(
                        data.size(0),       # B
                        1,                  # 1 new channel
                        data.size(2),       # H
                        data.size(3),       # W
                        device=device
                    )
                    # now data is (B, C+1, H, W)
            data = torch.cat([data, noise[:, 0:1, :, :]], dim=1)
            output = model(data)
            test_loss += loss_fn(output, target).item()  # Sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # Get the index of the max class score
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = correct / len(test_loader.dataset)

    print('Test set: Average loss: %.4f, Accuracy: %d/%d (%.4f)' %
          (test_loss, correct, len(test_loader.dataset),
          100. * test_accuracy))
    return test_loss, test_accuracy

In [9]:
model = nn.Sequential(
    nn.Conv2d(3, 6, 5), # (3, 32, 32) -> (6, 28, 28)
    nn.BatchNorm2d(num_features=6),
    nn.ReLU(),
    nn.MaxPool2d(2, 2), # (6, 28, 28) -> (6, 14, 14)
    nn.Conv2d(6, 16, 5), # (6, 14, 14) -> (16, 10, 10)
    nn.ReLU(),
    nn.MaxPool2d(2, 2), # (16, 10, 10) -> (16, 5, 5)
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.Dropout(p=0.1),
    nn.ReLU(),
    nn.Linear(84, 10),
)

In [13]:
model_energy = nn.Sequential(
    nn.Conv2d(4, 6, 5), # (3, 32, 32) -> (6, 28, 28)
    nn.BatchNorm2d(num_features=6),
    nn.ReLU(),
    nn.MaxPool2d(2, 2), # (6, 28, 28) -> (6, 14, 14)
    nn.Conv2d(6, 16, 5), # (6, 14, 14) -> (16, 10, 10)
    nn.ReLU(),
    nn.MaxPool2d(2, 2), # (16, 10, 10) -> (16, 5, 5)
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.Dropout(p=0.1),
    nn.ReLU(),
    nn.Linear(84, 10),
)

In [11]:
trained_model, train_loss_history, valid_loss_history, valid_accuracy_history = run_training_loop(model)

Train Epoch: 1  Average loss: 0.0508
Valid set: Average loss: 0.0421, Accuracy: 5241/10000 (52.4100)

Train Epoch: 2  Average loss: 0.0407
Valid set: Average loss: 0.0604, Accuracy: 3643/10000 (36.4300)

Train Epoch: 3  Average loss: 0.0368
Valid set: Average loss: 0.0415, Accuracy: 5291/10000 (52.9100)

Train Epoch: 4  Average loss: 0.0342
Valid set: Average loss: 0.0367, Accuracy: 5804/10000 (58.0400)

Train Epoch: 5  Average loss: 0.0323
Valid set: Average loss: 0.0342, Accuracy: 6088/10000 (60.8800)

Train Epoch: 6  Average loss: 0.0308
Valid set: Average loss: 0.0335, Accuracy: 6237/10000 (62.3700)

Train Epoch: 7  Average loss: 0.0294
Valid set: Average loss: 0.0531, Accuracy: 4717/10000 (47.1700)

Train Epoch: 8  Average loss: 0.0283
Valid set: Average loss: 0.0331, Accuracy: 6304/10000 (63.0400)

Train Epoch: 9  Average loss: 0.0274
Valid set: Average loss: 0.0343, Accuracy: 6291/10000 (62.9100)

Train Epoch: 10  Average loss: 0.0265
Valid set: Average loss: 0.0344, Accuracy: 6

In [36]:
trained_model, train_loss_history, valid_loss_history, valid_accuracy_history = run_training_loop_energy(model_energy, n_epochs=20)

Train Epoch: 1  Average loss: 0.0341
Valid set: Average loss: 0.0380, Accuracy: 5696/10000 (56.9600)

Train Epoch: 2  Average loss: 0.0340
Valid set: Average loss: 0.0389, Accuracy: 5636/10000 (56.3600)

Train Epoch: 3  Average loss: 0.0337
Valid set: Average loss: 0.0387, Accuracy: 5680/10000 (56.8000)

Train Epoch: 4  Average loss: 0.0336
Valid set: Average loss: 0.0382, Accuracy: 5725/10000 (57.2500)

Train Epoch: 5  Average loss: 0.0332
Valid set: Average loss: 0.0382, Accuracy: 5741/10000 (57.4100)

Train Epoch: 6  Average loss: 0.0331
Valid set: Average loss: 0.0381, Accuracy: 5812/10000 (58.1200)

Train Epoch: 7  Average loss: 0.0329
Valid set: Average loss: 0.0404, Accuracy: 5542/10000 (55.4200)

Train Epoch: 8  Average loss: 0.0329
Valid set: Average loss: 0.0391, Accuracy: 5714/10000 (57.1400)

Train Epoch: 9  Average loss: 0.0327
Valid set: Average loss: 0.0381, Accuracy: 5784/10000 (57.8400)

Train Epoch: 10  Average loss: 0.0326
Valid set: Average loss: 0.0380, Accuracy: 5

In [28]:
test_performance(model)

Test set: Average loss: 0.0352, Accuracy: 6146/10000 (61.4600)


(0.03515353546142578, 0.6146)

In [38]:
test_performance_energy(model_energy)

Test set: Average loss: 0.0392, Accuracy: 5631/10000 (56.3100)


(0.03919565867781639, 0.5631)